<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tiktoken

def chunk_text(
    text,
    chunk_size=200,
    overlap=30,
    model="text-embedding-3-small"
):
    tokenizer = tiktoken.encoding_for_model(model)
    tokens = tokenizer.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
        start += chunk_size - overlap

    return chunks

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()

def embed_texts(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [d.embedding for d in response.data]

In [ ]:
pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.0 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

class FaissVectorStore:
    def __init__(self, embedding_dim):
        self.index = faiss.IndexFlatL2(embedding_dim)
        self.texts = []

    def add(self, embeddings, texts):
        self.index.add(np.array(embeddings).astype("float32"))
        self.texts.extend(texts)

    def search(self, query_embedding, top_k=3):
        D, I = self.index.search(
            np.array([query_embedding]).astype("float32"),
            top_k
        )
        return [self.texts[i] for i in I[0]]

In [ ]:
def retrieve(query, vector_store, top_k=3):
    query_embedding = embed_texts([query])[0]
    passages = vector_store.search(query_embedding, top_k)
    return passages


In [ ]:
#text = open("dataset.txt").read()
text = """
Transformers are a class of deep learning architectures introduced by Vaswani et al.
in the landmark paper "Attention Is All You Need". Unlike traditional recurrent or
convolutional neural networks, transformers rely entirely on attention mechanisms
to model dependencies between tokens in a sequence. This design choice enables
parallel computation and significantly improves training efficiency.

The core building block of the transformer architecture is the self-attention
mechanism. Self-attention allows each token in a sequence to attend to all other
tokens and compute a weighted representation based on relevance. This is done
using queries, keys, and values derived from the input embeddings through learned
linear transformations.

The attention score between two tokens is computed as the dot product between the
query of one token and the key of another token. These scores are scaled and passed
through a softmax function to obtain attention weights. The weighted sum of values
produces the final attention output. This allows the model to focus on relevant
context dynamically for each position.

Multi-head attention extends this idea by running multiple attention mechanisms in
parallel. Each head attends to the input sequence from a different representation
subspace. The outputs from all heads are concatenated and projected back into the
model dimension. This improves the model’s capacity to capture diverse linguistic
patterns such as syntax, semantics, and long-range dependencies.

A transformer encoder is composed of multiple layers. Each layer consists of a
multi-head self-attention sub-layer followed by a position-wise feed-forward neural
network. Residual connections are applied around each sub-layer, and layer
normalization is used to stabilize training. Dropout is often applied for
regularization.

The feed-forward network inside a transformer layer operates independently on
each token position. It typically consists of two linear transformations with a
non-linear activation function such as ReLU or GELU in between. This component
increases the representational power of the model.

Position information is critical for transformers because attention alone does
not encode sequence order. To address this, positional encodings are added to the
input embeddings. These encodings can be either fixed sinusoidal functions or
learned embeddings. Positional encodings allow the model to understand relative
and absolute token positions.

The transformer decoder includes an additional component known as masked
self-attention. This prevents tokens from attending to future positions during
training, ensuring autoregressive behavior. The decoder also attends to encoder
outputs using cross-attention, allowing it to condition on the input sequence.

Transformers have become the foundation of modern natural language processing
models. BERT uses a bidirectional transformer encoder to learn contextualized
representations during pretraining. GPT models use a unidirectional transformer
decoder optimized for text generation. T5 reformulates all NLP tasks into a
text-to-text format using an encoder-decoder transformer.

Beyond NLP, transformers have been successfully applied to computer vision,
speech recognition, reinforcement learning, and protein modeling. Vision
Transformers treat image patches as tokens and apply standard transformer layers.
This demonstrates the generality and flexibility of attention-based architectures.

One of the major strengths of transformers is their ability to model long-range
dependencies without the vanishing gradient problems associated with recurrent
networks. However, the quadratic complexity of self-attention with respect to
sequence length remains a challenge. Various efficient transformer variants
have been proposed to address this limitation.

Overall, transformers represent a paradigm shift in sequence modeling. Their
scalability, expressiveness, and parallelism have enabled the development of
large language models with billions of parameters, powering applications such as
chatbots, code assistants, search engines, and automated reasoning systems.
"""


chunks = chunk_text(text)
embeddings = embed_texts(chunks)

store = FaissVectorStore(embedding_dim=len(embeddings[0]))
store.add(embeddings, chunks)

results = retrieve("Explain transformer attention", store, top_k=3)

for r in results:
    print(r)


Transformers are a class of deep learning architectures introduced by Vaswani et al.
in the landmark paper "Attention Is All You Need". Unlike traditional recurrent or
convolutional neural networks, transformers rely entirely on attention mechanisms
to model dependencies between tokens in a sequence. This design choice enables
parallel computation and significantly improves training efficiency.

The core building block of the transformer architecture is the self-attention
mechanism. Self-attention allows each token in a sequence to attend to all other
tokens and compute a weighted representation based on relevance. This is done
using queries, keys, and values derived from the input embeddings through learned
linear transformations.

The attention score between two tokens is computed as the dot product between the
query of one token and the key of another token. These scores are scaled and passed
through a softmax function to obtain attention weights. The weighted sum of values
produce